# Train · проверка очистки и удаления дублей

Источник этой версии — `train_text_v1`. Сначала заменяем визуально пустые символы обычным пробелом и убираем подтверждённые складские окончания. Затем группируем по полному очищенному описанию без учёта регистра, пробелов, отдельных знаков оформления Markdown и списков, точек и восклицательных знаков в конце. В каждой группе остаётся самая ранняя строка исходного JSONL. SEO-текст и метки сохраняются.

## 1. Сверяем исходники, результаты и хеши сборки

In [1]:
import csv
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dataset_review/build_train_dedup_v1.py").is_file()
)
sys.path.insert(0, str(root / "dataset_review"))

from build_train_dedup_v1 import (
    SEO_MARKER, VISUAL_SPACES, duplicate_key, duplicate_reason, label_signature,
    replace_visual_spaces, sha256, whitespace_key, without_stock,
)

version_dir = root / "data/processed/train_dedup_v1"
manifest = json.loads((version_dir / "manifest.json").read_text(encoding="utf-8"))
for name in ("raw_source", "source", "dataset", "removed_duplicates"):
    info = manifest[name]
    assert sha256(root / info["path"]) == info["sha256"], name
for path, digest in manifest["build_files_sha256"].items():
    assert sha256(root / path) == digest, path

with (root / manifest["raw_source"]["path"]).open(encoding="utf-8") as stream:
    raw = {number: json.loads(line) for number, line in enumerate(stream, 1)}
incoming = pq.read_table(root / manifest["source"]["path"])
result = pq.read_table(root / manifest["dataset"]["path"])
with (root / manifest["removed_duplicates"]["path"]).open(encoding="utf-8", newline="") as stream:
    removed = list(csv.DictReader(stream))

assert result.schema.equals(incoming.schema)
for name, records in (
    ("raw_source", raw), ("source", incoming), ("dataset", result), ("removed_duplicates", removed)
):
    assert len(records) == manifest[name]["records"], name

display(pd.DataFrame([manifest[name] for name in ("raw_source", "source", "dataset", "removed_duplicates")])
        [["path", "records", "sha256"]])
print("Схема Parquet, SHA-256 файлов и количество записей проверены.")

,path,records,sha256
0,data/raw/public-train-annotated.jsonl,10000,b7118d68086624fbeab43839f22a635d4e325e4ee339f2...
1,data/processed/train_text_v1/train.parquet,10000,74990784db9e4f47bb41b9d5d4fa083f44292c49f642a3...
2,data/processed/train_dedup_v1/train.parquet,4749,57d0704e82a31c0ce91ec74196652f328b3f79951a3536...
3,data/processed/train_dedup_v1/removed_duplicat...,5251,08f50bede5422351e66c51c30bde310ba5e56f6e5c5c04...


Схема Parquet, SHA-256 файлов и количество записей проверены.


## 2. Проверяем очистку всех 10 000 строк

In [2]:
before = {row["source_line"]: row for row in incoming.to_pylist()}
after = {row["source_line"]: row for row in result.to_pylist()}
assert len(before) == len(incoming) and len(after) == len(result)
assert before.keys() == raw.keys()
assert len({row["item_id"] for row in after.values()}) == len(after)

cleaned = {}
statistics = Counter()
blank_characters = Counter()
for number, row in before.items():
    assert row["item_id"] == raw[number]["item_id"], number
    assert row["found_microcategories"] == raw[number]["found_microcategories"], number

    original = row["description"]
    blank_characters.update(char for char in original if ord(char) in VISUAL_SPACES)
    with_spaces = replace_visual_spaces(original)
    description = without_stock(with_spaces)
    title = replace_visual_spaces(row["title"])
    assert description and title, number
    assert original == description or not any(ord(char) in VISUAL_SPACES for char in description), number
    assert without_stock(description) == description, number
    original_markers = [match.casefold() for match in SEO_MARKER.findall(original)]
    assert original_markers == [match.casefold() for match in SEO_MARKER.findall(description)], number

    statistics["Визуальные пробелы в описании"] += with_spaces != original
    statistics["Визуальные пробелы в заголовке"] += title != row["title"]
    statistics["Складские окончания"] += description != with_spaces
    statistics["SEO-маркер перед дедупликацией"] += bool(original_markers)
    cleaned[number] = dict(row, title=title, description=description)

spaces, stock, dedup = manifest["transformations"]
assert spaces["changed_description_rows"] == statistics["Визуальные пробелы в описании"]
assert spaces["changed_title_rows"] == statistics["Визуальные пробелы в заголовке"]
assert stock["changed_description_rows"] == statistics["Складские окончания"]
assert manifest["seo"]["rows_before"] == statistics["SEO-маркер перед дедупликацией"]
statistics["SEO-маркер после дедупликации"] = sum(
    bool(SEO_MARKER.search(row["description"])) for row in after.values()
)
assert manifest["seo"]["rows_after"] == statistics["SEO-маркер после дедупликации"]
display(pd.Series(statistics, name="Строк").to_frame())
print("Заменено визуально пустых символов:", sum(blank_characters.values()))
print({f"U+{ord(char):04X}": count for char, count in blank_characters.items()})
print("Все SEO-маркеры совпали по тексту и порядку до удаления копий.")

,Строк
Визуальные пробелы в описании,1212
Визуальные пробелы в заголовке,0
Складские окончания,69
SEO-маркер перед дедупликацией,1192
SEO-маркер после дедупликации,900


Заменено визуально пустых символов: 14709
{'U+2800': 14595, 'U+3164': 114}
Все SEO-маркеры совпали по тексту и порядку до удаления копий.


## 3. Подтверждаем, что для каждой копии сохранилось описание с теми же метками

In [3]:
groups = defaultdict(list)
for number, row in sorted(cleaned.items()):
    groups[duplicate_key(row["description"])].append(number)

expected_kept = {lines[0] for lines in groups.values()}
assert set(after) == expected_kept
assert all(after[number] == cleaned[number] for number in expected_kept)
assert len({duplicate_key(row["description"]) for row in after.values()}) == len(after)

report = {int(row["source_line"]): row for row in removed}
assert len(report) == len(removed)
assert set(report) == set(before) - expected_kept
for key, lines in groups.items():
    chosen = after[lines[0]]
    for number in lines[1:]:
        entry = report[number]
        assert int(entry["item_id"]) == before[number]["item_id"], number
        assert int(entry["kept_source_line"]) == lines[0], number
        assert int(entry["kept_item_id"]) == chosen["item_id"], number
        assert label_signature(cleaned[number]) == label_signature(chosen), number
        assert entry["reason"] == duplicate_reason(
            dict(cleaned[number], original_description=before[number]["description"]),
            dict(chosen, original_description=before[lines[0]]["description"]),
        ), number
assert Counter(row["reason"] for row in removed) == dedup["removed_by_reason_relative_to_kept"]
assert len(removed) == dedup["removed_rows"] == len(before) - len(after)

display(pd.DataFrame({
    "Проверка": [
        "Входные строки", "Точные описания во входном файле",
        "Точные описания после очистки", "Уникальные описания без учёта регистра и пробелов",
        "Осталось после удаления дублей", "Убрано копий", "Конфликты меток",
    ],
    "Строк": [
        len(before), len({row["description"] for row in before.values()}),
        len({row["description"] for row in cleaned.values()}),
        len({whitespace_key(row["description"]) for row in cleaned.values()}),
        len(after), len(removed), 0,
    ],
}))

label_counts = lambda rows: Counter(
    entry["mcId"] for row in rows for entry in row["found_microcategories"]
)
counts_before = label_counts(before.values())
counts_after = label_counts(after.values())
display(pd.DataFrame({
    "До": counts_before,
    "После": counts_after,
    "Доля до, %": {cat: round(count * 100 / len(before), 1) for cat, count in counts_before.items()},
    "Доля после, %": {cat: round(count * 100 / len(after), 1) for cat, count in counts_after.items()},
}).sort_index().rename_axis("mcId"))
print("Без меток до и после:", sum(not r["found_microcategories"] for r in before.values()),
      "→", sum(not r["found_microcategories"] for r in after.values()))

,Проверка,Строк
0,Входные строки,10000
1,Точные описания во входном файле,4830
2,Точные описания после очистки,4766
3,Уникальные описания без учёта регистра и пробелов,4759
4,Осталось после удаления дублей,4749
5,Убрано копий,5251
6,Конфликты меток,0


,До,После,"Доля до, %","Доля после, %"
mcId,,,,
102,5484,2423,54.8,51.0
103,5073,2294,50.7,48.3
104,4582,2252,45.8,47.4
105,4562,2236,45.6,47.1
106,5481,2591,54.8,54.6


Без меток до и после: 2909 → 1314


## 4. Смотрим примеры и проверяем строку по её номеру в исходном JSONL

In [4]:
examples = pd.DataFrame(removed).groupby("reason", sort=False).head(2)
display(examples.reset_index(drop=True))

for reason, text_before, text_after in (
    ("Визуальные пробелы", before[4]["description"], cleaned[4]["description"]),
    ("Складское окончание", before[1]["description"], cleaned[1]["description"]),
):
    length = 240
    print(reason)
    if reason == "Визуальные пробелы":
        print("До   ", repr(text_before[:length]))
        print("После", repr(text_after[:length]))
    else:
        print("До   ", repr(text_before[-length:]))
        print("После", repr(text_after[-length:]))
    print()

line_to_inspect = 78  # Можно указать любой номер строки исходного JSONL.
chosen_line = int(report[line_to_inspect]["kept_source_line"]) if line_to_inspect in report else line_to_inspect
row = after[chosen_line]
print(f"Строка {line_to_inspect} → сохранённая строка {chosen_line}, item_id={row['item_id']}")
print("Категории:", [entry["mcId"] for entry in row["found_microcategories"]])
print(row["description"][:450])

,source_line,item_id,kept_source_line,kept_item_id,reason
0,43,2520535251663,3,2514956500363,exact
1,45,2507179754823,27,2507186251879,exact
2,78,2512544750819,1,2512542250248,exact_after_cleanup
3,94,2505919000177,23,2517637750116,case_or_whitespace
4,122,2512543000672,1,2512542250248,exact_after_cleanup
5,152,2505917251373,23,2517637750116,case_or_whitespace
6,1471,2518017250416,48,2512321250382,formatting
7,1510,2504539000483,1219,2511184250414,formatting


Визуальные пробелы
До    'РЕМОНТ КВАРТИР ПОД КЛЮЧ И ПО ОТДЕЛЬНЫМ ПОМЕЩЕНИЯМ\nНовостройки и вторичное жильё\nКомнаты • Кухни • Коридоры • Санузлы\nЧастная бригада — от демонтажа до чистовой отделки\n⠀\nНЕ ОБЯЗАТЕЛЬНО НАЧИНАТЬ С РЕМОНТА ВСЕЙ КВАРТИРЫ\nМожно полностью подгот'
После 'РЕМОНТ КВАРТИР ПОД КЛЮЧ И ПО ОТДЕЛЬНЫМ ПОМЕЩЕНИЯМ\nНовостройки и вторичное жильё\nКомнаты • Кухни • Коридоры • Санузлы\nЧастная бригада — от демонтажа до чистовой отделки\n\nНЕ ОБЯЗАТЕЛЬНО НАЧИНАТЬ С РЕМОНТА ВСЕЙ КВАРТИРЫ\nМожно полностью подгото'

Складское окончание
До    ' перечня работ.\n\nЗвоните или пишите в сообщениях Авито. Ответим на вопросы и подготовим предварительный расчёт.\n\nМесто на складе: m-9245 Номер продаваемого оборудования: r-6717 Марка: s-8059 Размеры оборудования на фото: 163,7 *209,4 *109,2'
После 'ан помещения.\n\nВыезд на замер и консультация. Стоимость ремонта рассчитывается индивидуально после осмотра объекта и согласования перечня работ.\n\nЗвоните или пишите в сообщениях Авито. О